## Autism spectrum disorder (ASD) Prediction

<div>    
<img src="https://engage-education.com/ie/wp-content/uploads/sites/5/2019/04/istock_autism.jpg" width="700", align="center"/>    
</div>

[Image credit](https://engage-education.com/ie/blog/supporting-students-with-autism-in-the-classroom/#!)

According to CDC ([Center for Disease Control and Prevention](https://www.cdc.gov/ncbddd/autism/facts.html)), Autism spectrum disorder (ASD) is defined as: 

"Autism spectrum disorder (ASD) is a developmental disability  that can cause significant social, communication and behavioral challenges. There is often nothing about how people with ASD look that sets them apart from other people, but people with ASD may communicate, interact, behave, and learn in ways that are different from most other people. The learning, thinking, and problem-solving abilities of people with ASD can range from gifted to severely challenged. Some people with ASD need a lot of help in their daily lives; others need less.

A diagnosis of ASD now includes several conditions that used to be diagnosed separately: autistic disorder, pervasive developmental disorder not otherwise specified (PDD-NOS), and Asperger syndrome. These conditions are now all called autism spectrum disorder."

One of the community competitions (this one) offered us the opportunity to see autism with data. Let's explore.

**The dataset**: The dataset is composed of survey results of people who filled an app form. There are labels portraying whether the person received a diagnosis of autism. 

- **Objective**: Build machine learning models to predict the likelihood of having autism based on the given features.

- The **evaluation metric** for this competition is AUC-ROC Score. 


### Part-1: EDA

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
import seaborn as sns

import plotly.io as pio
import plotly.express as px
import plotly.figure_factory as ff
import plotly.graph_objects as go
from sklearn.preprocessing import LabelEncoder

from plotly.subplots import make_subplots
from plotly.offline import init_notebook_mode, iplot

init_notebook_mode(connected=True)
pio.templates.default = "none"

import warnings
warnings.filterwarnings('ignore')


PATH = '/kaggle/input/autism-prediction/Autism-prediction/'
train = pd.read_csv(PATH + "/train.csv")
test = pd.read_csv(PATH + "/test.csv")
sample_submission = pd.read_csv(PATH + "/sample_submission.csv")

### Shape of data
- Train data: 800 rows, 21 cols + target columns
- Test data: 200 roes, 21 cols

In [ ]:
display(train.shape)
display(test.shape)

### Look at the head of the test 

Below is the columns description of the datasets

- ID - ID of the patient
- A1_Score to A10_Score - Score based on Autism Spectrum Quotient (AQ) 10 item screening tool
- age - Age of the patient in years
- gender - Gender of the patient
- ethnicity - Ethnicity of the patient
- jaundice - Whether the patient had jaundice at the time of birth
- autism - Whether an immediate family member has been diagnosed with autism
- contry_of_res - Country of residence of the patient
- used_app_before - Whether the patient has undergone a screening test before
- result - Score for AQ1-10 screening test
- age_desc - Age of the patient
- relation - Relation of patient who completed the test
- Class/ASD - Classified result as 0 or 1. Here 0 represents No and 1 represents Yes. This is the target column, and during submission submit the values as 0 or 1 only.

In [ ]:
train.head()

### Null Values
- No null values in both train and test datasets

In [ ]:
display(train.isna().sum().sum())
display(test.isna().sum().sum())

### Some data cleaning

In [ ]:
# fix the spelling issue 
train = train.rename(columns = {'austim': 'autism', 'contry_of_res':'country_of_res'}, errors="raise")
test = test.rename(columns = {'austim': 'autism', 'contry_of_res':'country_of_res'}, errors="raise")

# fix the other/Other capitalization
train['country_of_res'][train['country_of_res'] == 'others'] = 'Others'
test['country_of_res'][test['country_of_res'] == 'others'] = 'Others'

train['ethnicity'][train['ethnicity'] == 'others'] = 'Others'
test['ethnicity'][test['ethnicity'] == 'others'] = 'Others'

In [ ]:
# drop un-important cols
train.drop(['ID', 'age_desc'], axis=1, inplace=True)
test.drop(['ID', 'age_desc'], axis=1, inplace=True)

cat_cols = [col for col in train.columns if train[col].dtype == 'object']
num_cols = [col for col in train.columns if train[col].dtype == 'int'][0:-1]

ASD_pos = train[train['Class/ASD']==1]
ASD_neg = train[train['Class/ASD']==0]

### Check for categorical features uniqueness
- Only country of residence columns has different unique values in train (61) and test (44) datasets.
- Watch this feature if label encoding is reruired in model training.

In [ ]:
for col in cat_cols:
    x = train[col].nunique() 
    y = test[col].nunique()
    print("{}: train {} unique, test {} unique".format(col, x, y))

In [ ]:
train['country_of_res'].unique()

### Features distribution
### Target : ASD Class
- Around 77% of the target is ASD negative (0) and 23% ASD positive (1)
- Target class is imbalanced! 


In [ ]:
label = ['ASD Positive (=1)', 'ASD Negative (=0)']
value = [ASD_pos.shape[0], ASD_neg.shape[0]] 
pct = [value[0]*100/len(train), value[1]*100/len(train)]


fig = go.Figure(data=[go.Bar(
            y=value, x=label,
            text=(np.round(pct,2)),
            textposition=['outside', 'inside'],
            texttemplate = ["<b style='color: #f'>%{text}%</b>"]*2,
            textfont=dict(  family="sans serif",
                            size=16,
                            color="black"),
            orientation='v',
            marker_color=['purple', 'lightsalmon'],
            opacity=1.0,
                    )])
fig.update_layout(title='<b>Target: ASD Class <b>', 
                  font_family="San Serif",
                  template= 'simple_white',
                  yaxis_linewidth=2.5,
                  width=600, 
                  height=400,
                  bargap=0.2,
                  barmode='group',
                  titlefont={'size': 20},
                  )
fig.update_xaxes(showgrid=False, showline=True)
fig.update_yaxes(showgrid=False, showline=False, showticklabels=False, ticks='')
fig.show()

### Gender
- Gender distribution is fairly balanced. 52% are Females (f) and the remaining 48% are males (m)
- 63% of ASD_positive are females.

In [ ]:
fig = go.Figure()
fig.add_trace(go.Histogram(x=ASD_pos['gender'],histnorm='percent',
              name='ASD_pos', marker_color = 'purple'),
              )
fig.add_trace(go.Histogram(x=ASD_neg['gender'],histnorm='percent',
              name='ASD_neg', marker_color = 'salmon', opacity=0.85),
             )  

fig.update_layout(title="Gender of the patient", 
                  font_family="San Serif",
                  titlefont={'size': 20},
                  template='simple_white',
                  width=600, 
                  height=400,
                  legend=dict(
                  orientation="v", y=1, yanchor="top", x=1.20, xanchor="right" )                 
                 ).update_xaxes(categoryorder='total descending')
fig.show()

### Ethnicity
- 11 ethnic categories are present in the data, including "?" (not willing to tell?/is it NA?) and others
- The majorities are White-European followed by the "?", Asian and Middle-Easters
- Among the ASD-positive subjects, White-European make up 58% while being only 26% of the total sample (train dataset). Asian (9%) and Latinos & "?" (both at 8%) come in second and third. 

In [ ]:
fig = px.histogram(train, x="ethnicity",
                   width=600, 
                   height=400,
                   histnorm='percent',
                   template="simple_white"
                   )

fig.update_layout(title="Ethnic group of the patient", 
                  font_family="San Serif",
                  titlefont={'size': 20},
                  legend=dict(
                  orientation="v", y=1, yanchor="top", x=1.0, xanchor="right" )                 
                 ).update_xaxes(categoryorder='total descending') # ordering the x-axis values
#custom color
colors = ['lightgray',] * 15  
colors[1] = 'crimson' 
colors[0] = 'lightseagreen' 


fig.update_traces(marker_color=colors, 
                )
fig.show()

fig = go.Figure()
fig.add_trace(go.Histogram(x=ASD_pos['ethnicity'],histnorm='percent',
              name='ASD_pos', marker_color = 'purple'),
              )
fig.add_trace(go.Histogram(x=ASD_neg['ethnicity'],histnorm='percent',
              name='ASD_neg', marker_color = 'salmon', opacity=0.85),
             )  

fig.update_layout(title="Ethnic group of the patient", 
                  font_family="San Serif",
                  titlefont={'size': 20},
                  template='simple_white',
                  width=600, 
                  height=400,
                  legend=dict(
                  orientation="v", y=1, yanchor="top", x=1.0, xanchor="right" )                 
                 ).update_xaxes(categoryorder='total descending') 

fig.show()

### Jaundice
- The majority of the patients had no jaundice at birth
- Of those who had jaundice at birth, the majority (123) had tested ASD_negative compared to the 73 positive cases. Jaundice at birth may not mean ASD positivity.

In [ ]:
fig = go.Figure()
fig.add_trace(go.Histogram(x=ASD_pos['jaundice'],histnorm='',
              name='ASD_pos', marker_color = 'purple'),
              )
fig.add_trace(go.Histogram(x=ASD_neg['jaundice'],histnorm='',
              name='ASD_neg', marker_color = 'salmon', opacity=0.85),
             )  

fig.update_layout(title="Whether the patient had jaundice at the time of birth", 
                  font_family="San Serif",
                  titlefont={'size': 20},
                  template='simple_white',
                  width=600, 
                  height=400,
                  legend=dict(
                  orientation="v", y=1, yanchor="top", x=1.0, xanchor="right" )                 
                 ).update_xaxes(categoryorder='total descending') 

fig.show()


### Autism in the family
- Most patients have no immediate family who have been diagnose for autism.
- However, those who have autistic family members are likely to be diagnose with autism. 72 out of 117 (i.e 62%) of the ASD_positive patients have family members with autism.

In [ ]:
fig = go.Figure()
fig.add_trace(go.Histogram(x=ASD_pos['autism'],histnorm='',
              name='ASD_pos', marker_color = 'purple'),
              )
fig.add_trace(go.Histogram(x=ASD_neg['autism'],histnorm='',
              name='ASD_neg', marker_color = 'salmon', opacity=0.85),
             )  

fig.update_layout(title="Whether an immediate family member has been diagnosed with autism", 
                  font_family="San Serif",
                  titlefont={'size': 20},
                  template='simple_white',
                  width=600, 
                  height=400,
                  legend=dict(
                  orientation="v", y=1, yanchor="top", x=1.0, xanchor="right" )                 
                 ).update_xaxes(categoryorder='total descending') 

fig.show()

### Country of Residence of the patient
- USA has the most patients in the dataset (train), UEA and New Zealand come in second and third. 
- Only USA had equal amount of patients in both target groups (ASD_positive and ASD_negative)

In [ ]:
fig = px.histogram(train, x="country_of_res", 
                   width=900, 
                   height=400,
                   histnorm='percent',
                   template="simple_white")
fig.update_layout(title="<b> Country of Residence of the patient <b>", 
                  font_family="San Serif",
                  titlefont={'size': 20},
                  legend=dict(
                  orientation="v", y=1, yanchor="top", x=1.0, xanchor="right" )                 
                 ).update_xaxes(categoryorder='total descending') 

colors = ['lightgray',] * 100 
colors[0] = 'lightseagreen' 

fig.update_traces(marker_color=colors, 
                 ).update_xaxes(categoryorder='total descending')
                   
fig.show()

fig = px.treemap(train, path=['country_of_res','Class/ASD'], color='Class/ASD',
                 color_continuous_scale='teal',               
)

fig.update_layout(#title="<b> Country of Residence of the patient<b>",
                  titlefont={'size': 20, 'family': "San Serif"},
                  height=500, width=1000,
                  template='simple_white',
                  autosize=False,
                  margin=dict(l=50,r=50,b=50, t=250,
                             ),
                 )
fig.update_layout(margin = dict(t=50, l=50, r=50, b=100))
fig.show()

### Previous screening test
- Only 35 patients out of 800 have undergone previous screening tests. 
- 7 of the 35 patients have tested ASD_positive. 

In [ ]:
fig = go.Figure()

fig.add_trace(go.Histogram(x=ASD_pos['used_app_before'],histnorm='',
              name='ASD_pos', marker_color = 'purple'),
              )
fig.add_trace(go.Histogram(x=ASD_neg['used_app_before'],histnorm='',
              name='ASD_neg', marker_color = 'salmon', opacity=0.85),
             )    
fig.update_layout(barmode='group')
fig.update_layout(title="Whether the patient has undergone a screening test before", 
                  font_family="San Serif",
                  titlefont={'size': 20},
                  template='simple_white',
                  width=600, 
                  height=400,
                  legend=dict(
                  orientation="v", y=1, yanchor="top", x=1.0, xanchor="right" )                 
                 ).update_xaxes(categoryorder='total descending') 

fig.show()

### Relation of patient who completed the test
- Most patients have done the tests themselves. Suprisingly, health care professional comes last. 

In [ ]:
fig = go.Figure()

fig.add_trace(go.Histogram(x=ASD_pos['relation'],histnorm='',
              name='ASD_pos', marker_color = 'purple'),
              )
fig.add_trace(go.Histogram(x=ASD_neg['relation'],histnorm='',
              name='ASD_neg', marker_color = 'salmon', opacity=0.85),
             )    
fig.update_layout(barmode='group')
fig.update_layout(title="Relation of patient who completed the test", 
                  font_family="San Serif",
                  titlefont={'size': 20},
                  template='simple_white',
                  width=600, 
                  height=400,
                  legend=dict(
                  orientation="v", y=1, yanchor="top", x=1.0, xanchor="right" )                 
                 ).update_xaxes(categoryorder='total descending') 

fig.show()

### Age Distribution
- ASD_positive: 
  > average age is 32; the min is 10.9 years; and the maximum is 71.1 years
- ASD_negative
  > average age is 27.6; the min is 9.6 years; and the maximum is 72.4 years


In [ ]:
group_labels = ['ASD_pos', 'ASD_neg']
fig = ff.create_distplot([ASD_pos['age'], ASD_neg['age']],
                         group_labels, 
                         show_hist=True, 
                         show_rug=True,
                         colors=['purple', 'salmon'],
                         )
fig.update_layout(title='<b>Age distribution<b>',
                  xaxis_title='Age',
                  yaxis_title='density',
                  titlefont={'size': 20},
                  font_family = 'San Serif',
                  width=700,height=500,
                  template="simple_white",
                  showlegend=True,
                  legend=dict(
                      orientation="v",
                      y=1, 
                      yanchor="top", 
                      x=1.0, 
                      xanchor="right",
                  )
                 )
fig.add_vrect(
    x0=9, x1=35,
    annotation_text="Young patients", annotation_position="top",
    fillcolor="lightgray", opacity=0.5,
    layer="below", line_width=0,
),
fig.show()

### Screening Test scores (QA1-10)
- The QA test scores looks binary (they are either 1 or 0). 
- Most of the questions except A8_score and (slightly) A1_scores, most of the screening question have fairly separated the positive and negative cases.


In [ ]:
fig = make_subplots(rows=5, cols=2,
                   subplot_titles=('A1_Score','A6_Score', 'A2_Score', 'A7_Score','A3_Score', 
                                   'A8_Score','A4_Score','A9_Score', 'A5_Score','A10_Score',
                                   ))     

for i, feat in enumerate(num_cols[0:5]):
    fig.add_trace(go.Histogram(x=ASD_pos[feat],histnorm='percent',
                  name='ASD_pos', marker_color = 'purple'),
                  row=i+1, col=1)
    fig.add_trace(go.Histogram(x=ASD_neg[feat],histnorm='percent',
                  name='ASD_neg', marker_color = 'salmon', opacity=0.85),

                 row=i+1, col=1)    
    fig.update_layout(barmode='overlay')
                           
for j, feat in enumerate(num_cols[5:12]):
    fig.add_trace(go.Histogram(x=ASD_pos[feat],histnorm='percent',
                  name='ASD_pos', marker_color = 'purple'),
                 row=j+1, col=2)
    fig.add_trace(go.Histogram(x=ASD_neg[feat],histnorm='percent',
                  name='ASD_neg', marker_color = 'salmon', opacity=0.85),
                 row=j+1, col=2)    
    fig.update_layout(barmode='overlay')

fig.update_layout(title=" <b> Screening test score (A1- A10) <b>",
                      font_family="San Serif",
                      titlefont={'size': 24},
                      width=900, height=1200,
                      template='simple_white',
                      showlegend=False,
                      bargap=0.1, 
                      bargroupgap=0.1
                     )

fig.update_layout(barmode='group')
fig.show()

<!-- AQ_score = ['A1_Score', 'A2_Score', 'A3_Score', 'A4_Score', 'A5_Score', 'A6_Score',
       'A7_Score', 'A8_Score', 'A9_Score', 'A10_Score'] 

for item in AQ_score:
    fig = px.histogram(train, 
                 x=item,  
                 color='Class/ASD',
                 barmode='group',
                 histnorm='percent',
                 template='simple_white',
                 width=400, height=330,
                 color_discrete_sequence=['purple', 'salmon'],                                   
                )
    fig.update_layout(title="Distribusion of : " + item,
                      font_family="San Serif",
                      titlefont={'size': 20},
                      template='simple_white',
                      showlegend=False,
                      bargap=0.1, 
                      bargroupgap=0.1
                     )
    fig.show() -->

### Screening Test Result
- The screening test scores results are for the ASD_negative cases seems normally distrubuted with a mean value of around 6.
- Whereas for distribution of the ASD_positive casea are left-skewed with mean value of 10.5.
- Higher QA test score means higher chances of being ASD_positive.

In [ ]:
ASD_pos_result = train[train['Class/ASD'] == 1]['result']
ASD_neg_result = train[train['Class/ASD'] == 0]['result']

fig = go.Figure()
fig.add_trace(go.Violin(x=ASD_pos_result, line_color='lightseagreen', name='ASD_positive', y0=0))
fig.add_trace(go.Violin(x=ASD_neg_result, line_color='red', name= 'ASD_negative', y0=0))

fig.update_traces(orientation='h', side='positive', meanline_visible=True)
fig.update_layout(xaxis_showgrid=False, xaxis_zeroline=False)

fig.update_layout(title='<b> Screening test result distribution (QA1-10) <b>',
                  font_family="San Serif",
                  xaxis_title='Result',
                  titlefont={'size': 20},
                  width=600,
                  height=400,
                  template="simple_white",
                  showlegend=True,
                  )
fig.update_yaxes(showgrid=False, showline=False, showticklabels=False)
fig.show()

### Correlation heatmap (numerical features)
- From the numerical features only `result` shows moderate correlation with target `Class/ASD`

In [ ]:
x = ['age', 'result', 'Class/ASD']
df = train[x]

corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=np.bool))
corr = corr.mask(mask)


fig = go.Figure(data= go.Heatmap(z=corr,
                                 x=corr.index.values,
                                 y=corr.columns.values,
                                 colorscale='Blues',                                  
                                 )
                )
fig.update_layout(title_text='<b>Correlation Heatmap (Numerical features)<b>',
                  font_family="San Serif",
                  title_x=0.5,
                  titlefont={'size': 20},
                  width=450, height=400,
                  xaxis_showgrid=False,
                  xaxis={'side': 'bottom'},
                  yaxis_showgrid=False,
                  yaxis_autorange='reversed',                   
                                    autosize=False,
                  margin=dict(l=150,r=50,b=150,t=70,pad=0),
                  )
fig.show()

### Correlation heatmap (categorical features)
- We use Cramer's V correlation coefficient.
- We see that `A3_Score`, `A6_Score`, `A9_Score` and `ethnicity` are moderately correlated with the `target (Class/ASD)`. 
- We also notice that `A1_Score` and `A8_Score` have weak correlation with the target.
- Previous participation in screening programme (`used_app_before`) has no correlation with any other features or target.

In [ ]:
# the cramers_v function is copied from https://towardsdatascience.com/the-search-for-categorical-correlation-a1cf7f1888c9

def cramers_v(x, y): 
    confusion_matrix = pd.crosstab(x,y)
    chi2 = stats.chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2/n
    r,k = confusion_matrix.shape
    phi2corr = max(0, phi2-((k-1)*(r-1))/(n-1))
    rcorr = r-((r-1)**2)/(n-1)
    kcorr = k-((k-1)**2)/(n-1)
    return np.sqrt(phi2corr/min((kcorr-1),(rcorr-1)))


def plot_carmersV_corr(df):
    rows= []
    for x in df:
        col = []
        for y in df :
            cramers =cramers_v(df[x], df[y])
            col.append(round(cramers,2))
        rows.append(col)

    cramers_results = np.array(rows)
    df_corr = pd.DataFrame(cramers_results, columns = df.columns, index = df.columns)

    mask = np.triu(np.ones_like(df_corr, dtype=np.bool))
    df_corr = df_corr.mask(mask)


    fig = go.Figure(data= go.Heatmap(z=df_corr,
                                     x=df_corr.index.values,
                                     y=df_corr.columns.values,
                                     colorscale='purples',                                  
                                     )
                    )
    fig.update_layout(title_text='<b>Correlation Heatmap (Categorical features) <b>',
                      font_family="San Serif",
                      title_x=0.5,
                      titlefont={'size': 20},
                      width=750, height=700,
                      xaxis_showgrid=False,
                      xaxis={'side': 'bottom'},
                      yaxis_showgrid=False,
                      yaxis_autorange='reversed',                   
                                        autosize=False,
                      margin=dict(l=150,r=50,b=150,t=70,pad=0),
                      )
    fig.show()
    
plot_carmersV_corr(train.drop(['age', 'result'], axis=1))

### Short conclusion (EDA)
- Target is imbalanced.
- One feature (country of residence) has different uniques values in train and test dataset.
- Having `Jaudice` at birth seem to have no major influence on autism. However, `family history` seem to have correlation with autism diagnose.


### Modeling
- Build simple models such Logistic regression, Random forest, etc
- Try Stacking, Voting classifiers

In [ ]:
from scipy.stats import uniform

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import StackingClassifier
from sklearn.ensemble import VotingClassifier


from sklearn.model_selection import RandomizedSearchCV, GridSearchCV


from sklearn.model_selection import cross_validate, StratifiedKFold, RepeatedStratifiedKFold
from sklearn.metrics import confusion_matrix, plot_confusion_matrix, classification_report
from sklearn.metrics import roc_auc_score 
from sklearn import metrics

from sklearn.preprocessing import LabelEncoder


In [ ]:
label_encoder = LabelEncoder()
train_le = train.copy()
test_le = test.copy()

for col in cat_cols:
        train_le[col] = label_encoder.fit_transform(train[col])
        test_le[col] = label_encoder.fit_transform(test[col]) 
        
train = train_le
test = test_le

In [ ]:
FEATURES = train.columns[0:-1]
TARGET = train.columns[-1]

X = train.loc[:, FEATURES]
y = train.loc[:, TARGET]

seed = 0
fold = 5

In [ ]:
def score(X, y, model, cv):
    scoring = ["roc_auc"]
    scores = cross_validate(
        model, X, y, scoring=scoring, cv=cv, return_train_score=True,
    )
    scores = pd.DataFrame(scores).T
    return scores.assign(
        mean = lambda x: x.mean(axis=1),
        std = lambda x: x.std(axis=1),
    )

skf = StratifiedKFold(n_splits=fold, shuffle=True, random_state=seed)

### Logistic regression model

<!-- # logistic = LogisticRegression(solver='saga', 
#                               tol=1e-5, max_iter=10000,
#                               random_state=0)
# distributions = dict(C=uniform(loc=0, scale=4),
#                     penalty=['l2', 'l1'])
# clf = RandomizedSearchCV(logistic, distributions, random_state=seed)
# search = clf.fit(X, y)
# search.best_params_ -->

In [ ]:
model_lr = LogisticRegression(solver='saga', 
                              tol=1e-5, max_iter=10000,
                              random_state=0,
                              C=0.22685190926977272,
                              penalty='l2',
                             )

scores = score(X, y, model_lr, cv=skf)
display(scores)

### ROC_AUC curve

In [ ]:
model_lr.fit(X, y)
metrics.plot_roc_curve(model_lr, X, y)
plt.title('Logistic regression: ROC_AUC curve')
plt.show();

#### Submission (LR)

In [ ]:
y_pred_lr = pd.Series(
    model_lr.predict_proba(test)[:, 1],
    index=test.index,
    name=TARGET,
)
sub = pd.DataFrame({'ID': sample_submission.ID, 'Class/ASD': y_pred_lr})
sub.to_csv("submission_lr.csv", index=False)

### ExtraTreeClassifier

In [ ]:
model_etc = ExtraTreesClassifier(
    n_estimators=1000,
    max_depth=2,
    random_state=0,
)
scores = score(X, y, model_etc, cv=skf)
display(scores)

In [ ]:
model_etc.fit(X, y)
metrics.plot_roc_curve(model_etc, X, y)
plt.title('Extra trees classifier: ROC_AUC curve')
plt.show();


In [ ]:
y_pred_etc = pd.Series(
    model_etc.predict_proba(test)[:, 1],
    index=test.index,
    name=TARGET,
)
sub = pd.DataFrame({'ID': sample_submission.ID, 'Class/ASD': y_pred_etc})
sub.to_csv("submission_etc.csv", index=False)

### Blended submission

In [ ]:
sub1 = pd.DataFrame({'ID': sample_submission.ID, 'Class/ASD':(0.35* y_pred_etc + 0.65* y_pred_lr)})
sub1.to_csv("submission_etc35_lr65.csv", index=False)

sub2 = pd.DataFrame({'ID': sample_submission.ID, 'Class/ASD': (0.5* y_pred_etc + 0.5* y_pred_lr)})
sub2.to_csv("submission_etc5_lr5.csv", index=False)

sub3 = pd.DataFrame({'ID': sample_submission.ID, 'Class/ASD': (0.4* y_pred_etc + 0.6* y_pred_lr)})
sub3.to_csv("submission_etc4_lr6.csv", index=False)

- See V6 for Stacking classifiers + Voting classifiers
- LB scores were worse than the LR (0.93939)